In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import joblib

# 1. 加载数据
file_path = "/home/zhanyu/experiment/data-hh/my/train_86i3lia5Jj4=_aggregated_output.csv"  # 替换为你的文件路径
data = pd.read_csv(file_path)

# 2. 特征工程
# 从日期字段提取年份、月份和日期
data['flt_year'] = pd.to_datetime(data['flt_date']).dt.year
data['flt_month'] = pd.to_datetime(data['flt_date']).dt.month
data['flt_day'] = pd.to_datetime(data['flt_date']).dt.day

# 删除原始日期字段
data.drop(columns=['flt_date'], inplace=True)

# 3. 数据预处理
# 对分类字段进行编码
label_encoders = {}
for col in ['flt_no', 'a', 'b', 'c', 'aircraft']:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# 检查并处理缺失值
data.fillna(data.mean(), inplace=True)

# 4. 特征和目标变量划分
X = data[['flt_year', 'flt_month', 'flt_day', 'flt_no', 'a', 'b', 'c', 'aircraft', 'ab_duration', 'bc_duration', 'ac_duration']]
y_ab_pax = data['ab_pax']
y_bc_pax = data['bc_pax']
y_ac_pax = data['ac_pax']

# 分别划分训练集和测试集
X_train, X_test, y_ab_train, y_ab_test = train_test_split(X, y_ab_pax, test_size=0.2, random_state=42)
_, _, y_bc_train, y_bc_test = train_test_split(X, y_bc_pax, test_size=0.2, random_state=42)
_, _, y_ac_train, y_ac_test = train_test_split(X, y_ac_pax, test_size=0.2, random_state=42)

# 5. 构建随机森林模型
rf_ab = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_ab.fit(X_train, y_ab_train)

rf_bc = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_bc.fit(X_train, y_bc_train)

rf_ac = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf_ac.fit(X_train, y_ac_train)

# 6. 模型评估
y_ab_pred = rf_ab.predict(X_test)
y_bc_pred = rf_bc.predict(X_test)
y_ac_pred = rf_ac.predict(X_test)

mse_ab = mean_squared_error(y_ab_test, y_ab_pred)
mse_bc = mean_squared_error(y_bc_test, y_bc_pred)
mse_ac = mean_squared_error(y_ac_test, y_ac_pred)

print(f"ab_pax MSE: {mse_ab}")
print(f"bc_pax MSE: {mse_bc}")
print(f"ac_pax MSE: {mse_ac}")

# 7. 保存模型
joblib.dump(rf_ab, "rf_ab_model.pkl")
joblib.dump(rf_bc, "rf_bc_model.pkl")
joblib.dump(rf_ac, "rf_ac_model.pkl")

print("模型已保存！")

# 8. 展示部分预测结果
results = pd.DataFrame({
    'flt_year': X_test['flt_year'],
    'flt_month': X_test['flt_month'],
    'flt_day': X_test['flt_day'],
    'ab_pax_actual': y_ab_test,
    'ab_pax_predicted': y_ab_pred,
    'bc_pax_actual': y_bc_test,
    'bc_pax_predicted': y_bc_pred,
    'ac_pax_actual': y_ac_test,
    'ac_pax_predicted': y_ac_pred
})

print("部分预测结果：")
print(results.head(10))


ab_pax MSE: 369.97342760638116
bc_pax MSE: 411.8717632873741
ac_pax MSE: 170.83457027563955
模型已保存！
部分预测结果：
     flt_year  flt_month  flt_day  ab_pax_actual  ab_pax_predicted  \
125      2023         12        9            104         81.038417   
51       2023          2       21             93         76.878731   
139      2023         12       23             99         77.510060   
19       2023          1       20             89         87.083750   
104      2023         11       18             47         63.576517   
12       2023          1       13            114        101.005000   
76       2023          3       18             91         70.835983   
31       2023          2        1             68         66.182611   
81       2023          3       23             80         68.240179   
9        2023          1       10            107        105.210179   

     bc_pax_actual  bc_pax_predicted  ac_pax_actual  ac_pax_predicted  
125             88         90.838107             3